## はじめに

このノートブックでは、GlacierStyle ECサイトの非構造化データに対するCortex Search Serviceを作成します。

**前提条件:**
- Part 1〜3 が実行済みであること（Gold層テーブルの作成完了）
- FAQドキュメントのCortex Search ServiceはSnowsight UIで別途作成済み

**主な処理内容:**
- 運営マニュアル用Cortex Search Serviceの作成
- 音声ログ要約用Cortex Search Serviceの作成
- SNS投稿分析用Cortex Search Serviceの作成
- 各Serviceの検索テスト

**Cortex Searchとは:**
- 非構造化データに対するハイブリッド検索（キーワード＋セマンティック）を実現
- 自動的にベクトル埋め込みを生成し、検索インデックスを構築
- Snowflake Intelligenceのツールとして連携可能

In [ ]:
-- ============================================================================
-- 環境設定
-- ============================================================================
-- 使用するウェアハウスとスキーマを設定
USE WAREHOUSE COMPUTE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

## 1. Cortex Search Service 概要

本ハンズオンでは、以下の4つのCortex Search Serviceを作成します。

**1. SEARCH_FAQ**
- 対象テーブル: GOLD_FAQ_DOCUMENTS
- 検索対象: FAQ本文
- 用途: よくある質問の検索（**UI作成**）

**2. SEARCH_OPERATION_MANUALS**
- 対象テーブル: GOLD_OPERATION_MANUALS
- 検索対象: マニュアル本文
- 用途: 業務マニュアルの検索

**3. SEARCH_VOICE_LOGS**
- 対象テーブル: GOLD_VOICE_LOGS
- 検索対象: 通話要約
- 用途: 顧客対応履歴の検索

**4. SEARCH_SNS_MENTIONS**
- 対象テーブル: GOLD_SNS_MENTIONS_ANALYZED
- 検索対象: SNS投稿本文
- 用途: 顧客の声の検索

**共通パラメータ:**
- 埋め込みモデル: `snowflake-arctic-embed-l-v2.0`
- ターゲットラグ: 1日（`'1 day'`）
- ウェアハウス: `COMPUTE_WH`

## 2. 対象データの確認

Cortex Search Serviceを作成する前に、対象となるGold層テーブルのデータを確認します。

In [ ]:
-- ============================================================================
-- 対象テーブルのレコード数確認
-- ============================================================================
SELECT 
    'GOLD_FAQ_DOCUMENTS' AS table_name,
    COUNT(*) AS record_count,
    'FAQドキュメント（UI作成）' AS description
FROM GOLD_FAQ_DOCUMENTS
UNION ALL
SELECT 
    'GOLD_OPERATION_MANUALS' AS table_name,
    COUNT(*) AS record_count,
    '運営マニュアル' AS description
FROM GOLD_OPERATION_MANUALS
UNION ALL
SELECT 
    'GOLD_VOICE_LOGS' AS table_name,
    COUNT(*) AS record_count,
    '音声ログ（要約付き）' AS description
FROM GOLD_VOICE_LOGS
UNION ALL
SELECT 
    'GOLD_SNS_MENTIONS_ANALYZED' AS table_name,
    COUNT(*) AS record_count,
    'SNS投稿分析済み' AS description
FROM GOLD_SNS_MENTIONS_ANALYZED
ORDER BY table_name;

### 2-1. 運営マニュアルのデータ構造確認

In [ ]:
-- ============================================================================
-- 運営マニュアルのサンプルデータ確認
-- ============================================================================
SELECT 
    DEPARTMENT,
    CHAPTER,
    SECTION,
    SUBSECTION,
    LEFT(CONTENT_CHUNK, 100) AS content_preview
FROM GOLD_OPERATION_MANUALS
LIMIT 5;

### 2-2. 音声ログのデータ構造確認

In [ ]:
-- ============================================================================
-- 音声ログのサンプルデータ確認
-- ============================================================================
SELECT 
    CALL_ID,
    CATEGORY,
    INQUIRY_CATEGORY,
    OVERALL_SENTIMENT,
    LEFT(TRANSCRIBED_TEXT_SUMMARY, 100) AS summary_preview
FROM GOLD_VOICE_LOGS
LIMIT 5;

### 2-3. SNS投稿のデータ構造確認

In [ ]:
-- ============================================================================
-- SNS投稿のサンプルデータ確認
-- ============================================================================
SELECT 
    POST_ID,
    PLATFORM,
    POST_CATEGORY,
    OVERALL_SENTIMENT,
    EXTRACTED_CATEGORY,
    LEFT(CONTENT, 100) AS content_preview
FROM GOLD_SNS_MENTIONS_ANALYZED
LIMIT 5;

## 3. Cortex Search Serviceの作成

各データソースに対してCortex Search Serviceを作成します。

### 3-1. 運営マニュアル用 Cortex Search Service

業務マニュアルの検索サービスを作成します。

**設定項目:**
- サービス名: `SEARCH_OPERATION_MANUALS`
- 対象テーブル: `GOLD_OPERATION_MANUALS`
- 検索列: `CONTENT_CHUNK`
- 属性列: `DEPARTMENT`, `CHAPTER`, `SECTION`, `SUBSECTION`
- ターゲットラグ: 1日
- 埋め込みモデル: `snowflake-arctic-embed-l-v2.0`

In [ ]:
-- ============================================================================
-- 運営マニュアル用 Cortex Search Service の作成
-- ============================================================================
CREATE OR REPLACE CORTEX SEARCH SERVICE SEARCH_OPERATION_MANUALS
  ON CONTENT_CHUNK
  ATTRIBUTES DEPARTMENT, CHAPTER, SECTION, SUBSECTION
  WAREHOUSE = COMPUTE_WH
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS (
    SELECT 
        RELATIVE_PATH,
        DEPARTMENT,
        CHAPTER,
        SECTION,
        SUBSECTION,
        CONTENT_CHUNK
    FROM GOLD_OPERATION_MANUALS
    WHERE CONTENT_CHUNK IS NOT NULL
      AND LENGTH(TRIM(CONTENT_CHUNK)) > 0
  );

### 3-2. 音声ログ要約用 Cortex Search Service

コールセンターの通話要約を検索するサービスを作成します。

**設定項目:**
- サービス名: `SEARCH_VOICE_LOGS`
- 対象テーブル: `GOLD_VOICE_LOGS`
- 検索列: `TRANSCRIBED_TEXT_SUMMARY`
- 属性列: `CALL_ID`, `CATEGORY`, `INQUIRY_CATEGORY`, `OVERALL_SENTIMENT`
- ターゲットラグ: 1日
- 埋め込みモデル: `snowflake-arctic-embed-l-v2.0`

In [ ]:
-- ============================================================================
-- 音声ログ要約用 Cortex Search Service の作成
-- ============================================================================
CREATE OR REPLACE CORTEX SEARCH SERVICE SEARCH_VOICE_LOGS
  ON TRANSCRIBED_TEXT_SUMMARY
  ATTRIBUTES CALL_ID, CATEGORY, INQUIRY_CATEGORY, OVERALL_SENTIMENT
  WAREHOUSE = COMPUTE_WH
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS (
    SELECT 
        CALL_ID,
        SCENARIO_ID,
        CATEGORY,
        INQUIRY_CATEGORY,
        OVERALL_SENTIMENT,
        AGENT_ID,
        CALL_DURATION_SEC,
        CALL_START_TIME,
        TRANSCRIBED_TEXT_SUMMARY
    FROM GOLD_VOICE_LOGS
    WHERE TRANSCRIBED_TEXT_SUMMARY IS NOT NULL
      AND LENGTH(TRIM(TRANSCRIBED_TEXT_SUMMARY)) > 0
  );

### 3-3. SNS投稿分析用 Cortex Search Service

SNS上の顧客の声を検索するサービスを作成します。

**設定項目:**
- サービス名: `SEARCH_SNS_MENTIONS`
- 対象テーブル: `GOLD_SNS_MENTIONS_ANALYZED`
- 検索列: `CONTENT`
- 属性列: `PLATFORM`, `POST_CATEGORY`, `OVERALL_SENTIMENT`, `EXTRACTED_CATEGORY`
- ターゲットラグ: 1日
- 埋め込みモデル: `snowflake-arctic-embed-l-v2.0`

In [ ]:
-- ============================================================================
-- SNS投稿分析用 Cortex Search Service の作成
-- ============================================================================
CREATE OR REPLACE CORTEX SEARCH SERVICE SEARCH_SNS_MENTIONS
  ON CONTENT
  ATTRIBUTES PLATFORM, POST_CATEGORY, OVERALL_SENTIMENT, EXTRACTED_CATEGORY
  WAREHOUSE = COMPUTE_WH
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS (
    SELECT 
        POST_ID,
        PLATFORM,
        POST_TYPE,
        USERNAME,
        DISPLAY_NAME,
        CONTENT,
        POSTED_AT,
        LIKES,
        RETWEETS,
        REPLIES,
        POST_CATEGORY,
        OVERALL_SENTIMENT,
        EXTRACTED_CATEGORY,
        EXTRACTED_PRODUCT_NAME
    FROM GOLD_SNS_MENTIONS_ANALYZED
    WHERE CONTENT IS NOT NULL
      AND LENGTH(TRIM(CONTENT)) > 0
  );

## 4. Cortex Search Service の確認

作成したCortex Search Serviceの一覧と状態を確認します。

In [ ]:
-- ============================================================================
-- Cortex Search Serviceの一覧確認
-- ============================================================================
SHOW CORTEX SEARCH SERVICES IN SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

In [ ]:
-- ============================================================================
-- 各サービスの詳細確認
-- ============================================================================
DESCRIBE CORTEX SEARCH SERVICE SEARCH_OPERATION_MANUALS;

## 5. 検索テスト

作成したCortex Search Serviceを使用して、実際に検索を行います。

**SEARCH_PREVIEW関数の結果をフラット化して見やすく表示します。**

### 5-1. 運営マニュアルの検索テスト

In [ ]:
-- ============================================================================
-- 運営マニュアル検索テスト: 返品対応手順
-- ============================================================================
WITH search_result AS (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_OPERATION_MANUALS',
        '{
            "query": "返品の手続き方法",
            "columns": ["DEPARTMENT", "CHAPTER", "SECTION", "CONTENT_CHUNK"],
            "limit": 3
        }'
    ) AS result_json
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY f.value:"@scores":reranker_score::FLOAT DESC) AS rank,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:DEPARTMENT::STRING AS department,
    f.value:CHAPTER::STRING AS chapter,
    f.value:SECTION::STRING AS section,
    LEFT(f.value:CONTENT_CHUNK::STRING, 200) AS content_preview
FROM search_result,
    LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- ============================================================================
-- 運営マニュアル検索テスト: クレーム対応（フィルター付き）
-- ============================================================================
WITH search_result AS (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_OPERATION_MANUALS',
        '{
            "query": "クレーム対応のエスカレーション",
            "columns": ["DEPARTMENT", "CHAPTER", "SECTION", "CONTENT_CHUNK"],
            "filter": {"@eq": {"DEPARTMENT": "カスタマーサポート部"}},
            "limit": 3
        }'
    ) AS result_json
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY f.value:"@scores":reranker_score::FLOAT DESC) AS rank,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:DEPARTMENT::STRING AS department,
    f.value:CHAPTER::STRING AS chapter,
    f.value:SECTION::STRING AS section,
    LEFT(f.value:CONTENT_CHUNK::STRING, 200) AS content_preview
FROM search_result,
    LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

### 5-2. 音声ログ要約の検索テスト

In [ ]:
-- ============================================================================
-- 音声ログ検索テスト: 配送遅延に関する問い合わせ
-- ============================================================================
WITH search_result AS (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_VOICE_LOGS',
        '{
            "query": "配送が遅れている",
            "columns": ["CALL_ID", "CATEGORY", "OVERALL_SENTIMENT", "TRANSCRIBED_TEXT_SUMMARY"],
            "limit": 3
        }'
    ) AS result_json
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY f.value:"@scores":reranker_score::FLOAT DESC) AS rank,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:CALL_ID::STRING AS call_id,
    f.value:CATEGORY::STRING AS category,
    f.value:OVERALL_SENTIMENT::STRING AS sentiment,
    LEFT(f.value:TRANSCRIBED_TEXT_SUMMARY::STRING, 200) AS summary_preview
FROM search_result,
    LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- ============================================================================
-- 音声ログ検索テスト: ネガティブな問い合わせ（フィルター付き）
-- ============================================================================
WITH search_result AS (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_VOICE_LOGS',
        '{
            "query": "商品の不具合",
            "columns": ["CALL_ID", "CATEGORY", "INQUIRY_CATEGORY", "OVERALL_SENTIMENT", "TRANSCRIBED_TEXT_SUMMARY"],
            "filter": {"@eq": {"OVERALL_SENTIMENT": "negative"}},
            "limit": 3
        }'
    ) AS result_json
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY f.value:"@scores":reranker_score::FLOAT DESC) AS rank,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:CALL_ID::STRING AS call_id,
    f.value:CATEGORY::STRING AS category,
    f.value:INQUIRY_CATEGORY::STRING AS inquiry_category,
    f.value:OVERALL_SENTIMENT::STRING AS sentiment,
    LEFT(f.value:TRANSCRIBED_TEXT_SUMMARY::STRING, 200) AS summary_preview
FROM search_result,
    LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

### 5-3. SNS投稿の検索テスト

In [ ]:
-- ============================================================================
-- SNS投稿検索テスト: デスクライトに関する口コミ
-- ============================================================================
WITH search_result AS (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_SNS_MENTIONS',
        '{
            "query": "デスクライト おすすめ",
            "columns": ["POST_ID", "PLATFORM", "POST_CATEGORY", "OVERALL_SENTIMENT", "CONTENT"],
            "limit": 5
        }'
    ) AS result_json
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY f.value:"@scores":reranker_score::FLOAT DESC) AS rank,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:POST_ID::STRING AS post_id,
    f.value:PLATFORM::STRING AS platform,
    f.value:POST_CATEGORY::STRING AS post_category,
    f.value:OVERALL_SENTIMENT::STRING AS sentiment,
    LEFT(f.value:CONTENT::STRING, 150) AS content_preview
FROM search_result,
    LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- ============================================================================
-- SNS投稿検索テスト: ポジティブな投稿（フィルター付き）
-- ============================================================================
WITH search_result AS (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_SNS_MENTIONS',
        '{
            "query": "品質が良い 満足",
            "columns": ["POST_ID", "PLATFORM", "POST_CATEGORY", "EXTRACTED_PRODUCT_NAME", "CONTENT"],
            "filter": {"@eq": {"OVERALL_SENTIMENT": "positive"}},
            "limit": 5
        }'
    ) AS result_json
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY f.value:"@scores":reranker_score::FLOAT DESC) AS rank,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:POST_ID::STRING AS post_id,
    f.value:PLATFORM::STRING AS platform,
    f.value:POST_CATEGORY::STRING AS post_category,
    f.value:EXTRACTED_PRODUCT_NAME::STRING AS product_name,
    LEFT(f.value:CONTENT::STRING, 150) AS content_preview
FROM search_result,
    LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- ============================================================================
-- SNS投稿検索テスト: プラットフォーム別検索
-- ============================================================================
WITH search_result AS (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_SNS_MENTIONS',
        '{
            "query": "GlacierStyle インテリア",
            "columns": ["POST_ID", "PLATFORM", "USERNAME", "EXTRACTED_CATEGORY", "CONTENT"],
            "filter": {"@eq": {"PLATFORM": "instagram"}},
            "limit": 5
        }'
    ) AS result_json
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY f.value:"@scores":reranker_score::FLOAT DESC) AS rank,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:POST_ID::STRING AS post_id,
    f.value:PLATFORM::STRING AS platform,
    f.value:USERNAME::STRING AS username,
    f.value:EXTRACTED_CATEGORY::STRING AS category,
    LEFT(f.value:CONTENT::STRING, 150) AS content_preview
FROM search_result,
    LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

## まとめ

このノートブックでは、GlacierStyle ECサイトの非構造化データに対するCortex Search Serviceを作成しました。

### 作成したCortex Search Service

- **SEARCH_FAQ**: FAQドキュメント → よくある質問への回答検索（**UI作成**）
- **SEARCH_OPERATION_MANUALS**: 運営マニュアル → 業務手順・対応方法の検索
- **SEARCH_VOICE_LOGS**: 音声ログ要約 → 過去の顧客対応事例の検索
- **SEARCH_SNS_MENTIONS**: SNS投稿 → 顧客の声（VoC）の検索

### Cortex Searchの特長

- **ハイブリッド検索**: キーワードマッチとセマンティック類似度を組み合わせた高精度な検索
- **属性フィルター**: カテゴリ、感情、プラットフォーム等での絞り込み
- **自動更新**: ターゲットラグに基づく検索インデックスの自動更新
- **Intelligence連携**: Snowflake Intelligenceのツールとして利用可能

### 次のステップ

- **Part 5**: Snowflake Intelligenceによる自然言語分析
  - Semantic View + Cortex Searchを活用した対話型分析
  - VoC分析、売上分析、広告効果分析の実践